# MuSiQue-Ans dataset preparation

Use the original MuSiQue-Ans dev candidates, randomly sample 500 questions (seed 42), and cap each context at 6000 Qwen3.8 tokenizer tokens. Keep all source candidate paragraphs in order, including distractors. The chosen Qwen3.8-27B NF4/FP16 1-shot rater labels only question/context; answers and source metadata are retained separately for later answer evaluation.

See [README](README.md) for server commands. Prepared contexts already exist locally. **Model labels have not yet been generated.** Invalid model labels remain null and require review; they are not forced to a class.

This follows the paper's workflow, substituting our selected model and prompt. It does not reproduce the original Gemini labels. Official data: https://github.com/StonyBrookNLP/musique (CC BY 4.0).

In [1]:
from pathlib import Path
import json
from collections import Counter
from statistics import median, mean

cwd = Path.cwd().resolve()
PROJECT_ROOT = next((p for p in [cwd, *cwd.parents] if (p / "src/musique/prepare_dataset.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Start inside the project directory.")
DATA_DIR = PROJECT_ROOT / "data/musique/processed/dev500_seed42_ctx6000"
RESULTS_DIR = PROJECT_ROOT / "results/musique/dev500_seed42_ctx6000/qwen3.8_1shot"
manifest = json.loads((DATA_DIR / "manifest.json").read_text(encoding="utf-8"))
rows = [json.loads(line) for line in (DATA_DIR / "prepared.jsonl").read_text(encoding="utf-8").splitlines() if line.strip()]
assert len(rows) == len({row["id"] for row in rows}) == 500
assert all(row["context_tokens"] <= 6000 for row in rows)
print("Questions:", len(rows))
print("Paragraph-count distribution:", dict(Counter(row["n_paragraphs"] for row in rows)))
lengths = [row["context_tokens"] for row in rows]
print("Context tokens: min / median / mean / max =", min(lengths), median(lengths), round(mean(lengths), 1), max(lengths))
print("Truncated:", sum(row["truncated"] for row in rows))
print("Original answerable flags:", dict(Counter(row["source_answerable"] for row in rows)))
print("Pinned model/tokenizer revision:", manifest["model_revision"])
print("Demonstration ID:", manifest["rater_spec"]["demonstration"]["condition_id"])

Questions: 500
Paragraph-count distribution: {20: 498, 18: 1, 19: 1}
Context tokens: min / median / mean / max = 1340 2603.5 2649.2 5031
Truncated: 0
Original answerable flags: {True: 500}
Pinned model/tokenizer revision: 1d4bf0f2ff6012fd82039f2fa52739d0dd7c60c0
Demonstration ID: hotpot_1494_suf


## Inspect an example

Gold answer and original answerability below are inspection metadata, not autorater inputs. The final dataset will contain automatically predicted sufficiency, not human gold sufficiency.

In [2]:
example = rows[0]
print("ID:", example["id"])
print("QUESTION:", example["question"])
print("ANSWER (not shown to rater):", example["answer"])
print("CONTEXT PREVIEW:\n", example["context"][:1800])

ID: 2hop__35445_22458
QUESTION: What is the full name of the organization that publishes he Laws of the Game?
ANSWER (not shown to rater): International Federation of Association Football
CONTEXT PREVIEW:
 [Document 1] Willamette Law Review
The Willamette Law Review is a law review academic journal published by Willamette University College of Law in Salem, Oregon, United States. Founded in 1959 as a predecessor to an earlier publication, the triannual publication is housed in the Oregon Civic Justice Center. The journal is edited by students of the law school with oversight by the college's faculty. As of 2016, the "Willamette Law Review" has published a total of 52 volumes.

[Document 2] Groove Games
Groove Games was a Canadian developer and publisher of video game. Established in 2001, Groove Games had published more than 20 retail game titles on the PC, Xbox, and PlayStation 2 platforms.

[Document 3] Georgetown Journal of International Law
The Georgetown Journal of International L

## Label on the server

```bash
cd ~/sc4001_project
# src/musique and src/autorater are required.
# If raw + processed files have been uploaded, no Google Drive download is needed.
sbatch src/musique/run_musique_tc1.sh
```

Download results/musique back to this project. The following cell reads the run status; it does not infer labels or trigger model loading. Random human audit data is exported separately after labeling.

In [3]:
summary_path = RESULTS_DIR / "summary.json"
if summary_path.exists():
    print(json.dumps(json.loads(summary_path.read_text(encoding="utf-8")), indent=2))
else:
    print("Not labelled yet: 500 contexts are prepared; submit the server labeling job.")

Not labelled yet: 500 contexts are prepared; submit the server labeling job.
